# SSCD cosine-distance threshold on crystal0801

This notebook calibrates the production gate `MAX_COSINE_DISTANCE = 0.10` (`services/ai_worker/const/pg.py`) against the [crystal0801 near-duplicate dataset](https://www.kaggle.com/datasets/crystal0801/near-duplicate-dataset): 455 retail JPGs of one object shot from a slightly different angle or time. The set is not licensed for commercial use.

Ground truth is the filename `{base} ({duplicate}).jpg`. Images that share `base` are near-duplicates of the same object. Images with different base numbers are different objects. There is no annotator matrix.

The photos live in `.datasets/crystal0801-near-duplicate-dataset`.

Embeddings use the same SSCD Disc Mixup checkpoint as the worker (`m3/sscd-copy-detection`, `sscd_disc_mixup.torchscript.pt`). The paper name `facebook/sscd_disc_mixup` refers to this checkpoint. Distance is `scipy.spatial.distance.cosine`, which matches pgvector `<=>` on L2-normalized descriptors: a pair is inside the gate when the distance is **≤ 0.10**.

Run it from the AI worker environment so Torch and the embedder import:

```bash
cd services/ai_worker
uv run --with scipy --with matplotlib --with seaborn --with tqdm --with pandas jupyter lab ../../scripts/tune_threshold.ipynb
```


In [1]:
from __future__ import annotations

import re
import sys
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cosine
from tqdm.auto import tqdm


def repo_root() -> Path:
    """Returns the repository root that contains the AI worker package."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        worker = candidate / "services" / "ai_worker" / "embeddings" / "sscd.py"
        if worker.is_file():
            return candidate
    raise FileNotFoundError("run this notebook from the s3-dedup-engine repository")


ROOT = repo_root()
DATASETS = ROOT / ".datasets"
sys.path.insert(0, str(ROOT / "services" / "ai_worker"))

from embeddings.sscd import get_embedder  # noqa: E402

MAX_COSINE_DISTANCE = 0.10

MAX_PAIRS = 200
SEED = 7
DATASET_ROOT = DATASETS / "crystal0801-near-duplicate-dataset"
NAME_PATTERN = re.compile(r"^(\d+) \((\d+)\)(?: clone)?$", re.IGNORECASE)

embedder = get_embedder()
embedder.load()
print(f"threshold {MAX_COSINE_DISTANCE:.2f}  datasets {DATASETS}")

C:\Users\Kacper\AppData\Local\uv\cache\archive-v0\Z6djw8qWrwc1T_NWKsyF5\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Kacper\Desktop\s3-dedup-engine\services\ai_worker\.venv\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


threshold 0.10  datasets C:\Users\Kacper\Desktop\s3-dedup-engine\.datasets


In [2]:
def embed_file(path: Path, cache: dict[str, np.ndarray]) -> np.ndarray:
    """Returns a cached unit-norm SSCD descriptor for one image file."""
    key = str(path)
    cached = cache.get(key)
    if cached is None:
        with path.open("rb") as handle:
            cached = np.asarray(embedder.embed(handle), dtype=np.float64)
        cache[key] = cached
    return cached


def pair_distance(left: Path, right: Path, cache: dict[str, np.ndarray]) -> float:
    """Returns the SSCD cosine distance between two image files."""
    return float(cosine(embed_file(left, cache), embed_file(right, cache)))


LEFT_IMAGE = ""
RIGHT_IMAGE = ""
if LEFT_IMAGE and RIGHT_IMAGE:
    manual = pair_distance(Path(LEFT_IMAGE), Path(RIGHT_IMAGE), {})
    verdict = "inside gate" if manual <= MAX_COSINE_DISTANCE else "outside gate"
    print(
        f"{LEFT_IMAGE} vs {RIGHT_IMAGE}: cosine distance {manual:.6f} ({verdict}, threshold {MAX_COSINE_DISTANCE:.2f})"
    )

In [3]:
def grouped_photos(root: Path) -> dict[str, list[Path]]:
    """Groups {base} ({duplicate}).jpg files by the shared object id."""
    if not root.is_dir():
        raise FileNotFoundError(
            f"missing {root}. Unzip the crystal0801 near-duplicate set into .datasets."
        )
    groups: dict[str, list[Path]] = defaultdict(list)
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() != ".jpg":
            continue
        match = NAME_PATTERN.match(path.stem)
        if match is None:
            continue
        groups[match.group(1)].append(path)
    if not groups:
        raise FileNotFoundError(f"no {{base}} ({{duplicate}}).jpg files under {root}")
    for photos in groups.values():
        photos.sort(key=lambda item: item.name)
    return dict(groups)


def collect_pairs(
    groups: dict[str, list[Path]],
) -> tuple[list[tuple[Path, Path]], list[tuple[Path, Path]]]:
    """Builds within-object near-duplicates and across-object different pairs."""
    duplicates = [
        pair for photos in groups.values() for pair in combinations(photos, 2)
    ]
    bases = list(groups)
    different: list[tuple[Path, Path]] = []
    for left_index, left_base in enumerate(bases):
        for right_base in bases[left_index + 1 :]:
            for left in groups[left_base]:
                for right in groups[right_base]:
                    different.append((left, right))
    return duplicates, different


def sample_pairs(
    pairs: list[tuple[Path, Path]], limit: int, rng: np.random.Generator
) -> list[tuple[Path, Path]]:
    """Returns up to limit pairs in a reproducible random order."""
    if not pairs:
        return []
    order = rng.permutation(len(pairs))
    return [pairs[int(index)] for index in order[:limit]]


dataset_root = DATASET_ROOT
groups = grouped_photos(dataset_root)
photo_count = sum(len(photos) for photos in groups.values())
print(
    f"dataset {dataset_root.name}: {photo_count} photos in {len(groups)} object groups"
)

duplicates, different = collect_pairs(groups)
rng = np.random.default_rng(SEED)
duplicate_sample = sample_pairs(duplicates, MAX_PAIRS, rng)
different_sample = sample_pairs(different, MAX_PAIRS, rng)
print(
    f"near-duplicate pairs (same base): {len(duplicates)} (using {len(duplicate_sample)})"
)
print(
    f"different pairs (different base): {len(different)} (using {len(different_sample)})"
)

dataset crystal0801-near-duplicate-dataset: 456 photos in 64 object groups
near-duplicate pairs (same base): 2113 (using 200)
different pairs (different base): 101627 (using 200)


In [4]:
cache: dict[str, np.ndarray] = {}
rows: list[dict[str, object]] = []
for label, sample in (
    ("near-duplicate", duplicate_sample),
    ("different", different_sample),
):
    for left, right in tqdm(sample, desc=label):
        distance = pair_distance(left, right, cache)
        rows.append(
            {
                "label": label,
                "cosine_distance": distance,
                "left": left.name,
                "right": right.name,
            }
        )

frame = pd.DataFrame(rows)
print(f"embedded {len(cache)} unique photos")
for label, group in frame.groupby("label"):
    distances = group["cosine_distance"]
    inside = float((distances <= MAX_COSINE_DISTANCE).mean()) if len(distances) else 0.0
    print(
        f"{label} sample: n={len(distances)} of at most {MAX_PAIRS} mean={distances.mean():.4f} "
        f"median={distances.median():.4f} p90={distances.quantile(0.9):.4f} "
        f"sample fraction<= {MAX_COSINE_DISTANCE:.2f}: {inside:.1%}"
    )
print(
    frame.groupby("label")[["left", "right", "cosine_distance"]]
    .head(5)
    .to_string(index=False)
)

near-duplicate:   0%|          | 0/200 [00:00<?, ?it/s]

near-duplicate:   0%|          | 1/200 [00:01<03:33,  1.07s/it]

near-duplicate:   1%|          | 2/200 [00:01<02:01,  1.63it/s]

near-duplicate:   2%|▏         | 3/200 [00:01<01:32,  2.13it/s]

near-duplicate:   2%|▏         | 4/200 [00:01<01:18,  2.49it/s]

...

different:  98%|█████████▊| 195/200 [00:21<00:00, 13.07it/s]

different:  98%|█████████▊| 197/200 [00:21<00:00, 13.26it/s]

different: 100%|█████████▉| 199/200 [00:22<00:00, 10.98it/s]

different: 100%|██████████| 200/200 [00:22<00:00,  8.99it/s]

embedded 377 unique photos
different sample: n=200 of at most 200 mean=0.8850 median=0.8939 p90=0.9867 sample fraction<= 0.10: 0.0%
near-duplicate sample: n=200 of at most 200 mean=0.3455 median=0.3355 p90=0.4832 sample fraction<= 0.10: 1.5%
       left       right  cosine_distance
 19 (2).jpg  19 (6).jpg         0.388475
58 (18).jpg  58 (6).jpg         0.222130
 42 (4).jpg  42 (8).jpg         0.407047
 56 (6).jpg  56 (9).jpg         0.312581
62 (15).jpg  62 (5).jpg         0.443815
 1 (18).jpg  61 (7).jpg         0.771438
 48 (8).jpg 56 (17).jpg         0.864486
 56 (5).jpg  60 (1).jpg         0.815349
 25 (7).jpg  32 (1).jpg         0.735281
56 (16).jpg 58 (11).jpg         0.791726


In [5]:
if frame.empty:
    raise RuntimeError(
        "no crystal0801 pairs were sampled; check the {base} ({duplicate}).jpg names"
    )

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(
    data=frame,
    x="cosine_distance",
    hue="label",
    bins=30,
    element="step",
    stat="density",
    common_norm=False,
    ax=ax,
)
ax.axvline(MAX_COSINE_DISTANCE, color="#b00020", linestyle="--", linewidth=1.5)
ax.annotate(
    f"threshold {MAX_COSINE_DISTANCE:.2f}",
    xy=(MAX_COSINE_DISTANCE, ax.get_ylim()[1]),
    xytext=(6, -8),
    textcoords="offset points",
    color="#b00020",
    va="top",
)
ax.set_xlabel("SSCD cosine distance")
ax.set_ylabel("Density")
ax.set_title("crystal0801: SSCD cosine distance vs. 0.10 gate")
fig.tight_layout()
plt.show()

C:\Users\Kacper\AppData\Local\Temp\ipykernel_4628\3038896069.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## How to read the figure

The dashed line is the production gate. A pair at or to the left of it would be treated as the same image by pgvector.

- **Near-duplicate recall (sample)** is the fraction of the sampled same-base pairs, at most 200, with distance ≤ 0.10. Mass to the right of the line is a same-object pair the gate would keep as two images.
- **False-positive rate (sample)** is the fraction of the sampled different-base pairs, at most 200, with distance ≤ 0.10. Mass to the left of the line is a different object the gate would merge.

Both rates are estimates on that sample (`MAX_PAIRS`, `SEED = 7`), not on every crystal0801 pair. The 0.10 gate is supported for this collection when the near-duplicate histogram sits against 0 and the unrelated histogram sits clearly to the right of the line. Overlap is the reason to move the constant in `const/pg.py` and re-run this notebook on the same sample.
